# PDB SIFTS Derived Data Pipeline

Downloads all 16 EBI SIFTS mapping files via FTP and uploads them to the
Lakehouse bronze layer, archiving the previous version of each file whenever
its content has changed.

## What is SIFTS?
SIFTS (Structure Integration with Function, Taxonomy and Sequences) provides
cross-reference mappings between PDB chains and UniProt, Pfam, GO, CATH,
SCOP, Ensembl, taxonomy, and other databases.

## Available files
All files live under `ftp://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/tsv/`.
The full list is in `cdm_data_loaders.sifts.download.ALL_SIFTS_FILES`.
Set `FILES` in the configure cell to a list of filenames to download a subset.

## Output paths
| Object | S3 path |
|--------|--------|
| Current file | `s3://{LAKEHOUSE_BUCKET}/{LAKEHOUSE_KEY_PREFIX}/derived_data/sifts/{filename}` |
| Archived (on change) | `s3://{LAKEHOUSE_BUCKET}/{LAKEHOUSE_KEY_PREFIX}/derived_data/archive/{YYYY-MM-DD}/sifts/{filename}` |


In [1]:
"""Imports."""

from cdm_data_loaders.sifts.download import ALL_SIFTS_FILES
from cdm_data_loaders.sifts.run import run_sifts
from cdm_data_loaders.sifts.settings import SiftsSettings

Logging config not found at /home/user/git-repos/cdm-data-loaders/notebooks/logging_config.json (current working directory). Trying next source.
No logging config file found. Falling back to built-in config.


In [2]:
"""Configure the SIFTS pipeline.

Set DRY_RUN = True to log what would happen without downloading or uploading.
Set FILES to a list of filenames to download only a subset; None downloads all.
"""

# S3 bucket for the Lakehouse bronze layer
# format: bucket name (no s3:// scheme)
LAKEHOUSE_BUCKET = "cdm-lake"

# S3 key prefix for PDB datasets
# format: S3 key prefix (no leading/trailing slashes required)
LAKEHOUSE_KEY_PREFIX = "tenant-general-warehouse/kbase/datasets/pdb"

# SIFTS files to download. None = download all available files (ALL_SIFTS_FILES).
# Example: FILES = ["pdb_chain_uniprot.tsv.gz", "pdb_chain_go.tsv.gz"]
FILES: list[str] | None = None

# Set to True to skip all downloads and uploads (log-only mode)
DRY_RUN = False

settings = SiftsSettings(
    lakehouse_bucket=LAKEHOUSE_BUCKET,
    lakehouse_key_prefix=LAKEHOUSE_KEY_PREFIX,
    sifts_files=FILES,
    dry_run=DRY_RUN,
)
print(settings.model_dump())

{'lakehouse_bucket': 'cdm-lake', 'lakehouse_key_prefix': 'tenant-general-warehouse/kbase/datasets/pdb', 'sifts_ftp_host': 'ftp.ebi.ac.uk', 'sifts_files': None, 'dry_run': False}


In [3]:
"""Configure S3 credentials (use for local testing against the MinIO test container).

Set PROVIDE_CREDENTIALS = True to override the default credential chain and
point at a local MinIO instance instead of AWS.  Leave False when running in
an environment that already has credentials (IAM role, environment variables,
~/.aws/credentials, etc.).
"""

from cdm_data_loaders.utils.s3 import get_s3_client, reset_s3_client

PROVIDE_CREDENTIALS = True  # Set to True to use the credentials below instead of environment credentials
if PROVIDE_CREDENTIALS:
    reset_s3_client()  # Clear any existing client to ensure new credentials are used
    get_s3_client(
        args={
            "endpoint_url": "http://localhost:9000",
            "aws_access_key_id": "minioadmin",
            "aws_secret_access_key": "minioadmin",
        }
    )
    print("Using local MinIO credentials")
else:
    print("Using environment / IAM credentials")

Using local MinIO credentials


In [4]:
"""Run the SIFTS pipeline."""

result = run_sifts(settings)

print(f"Dry run : {result.dry_run}")
print(f"Files   : {len(result.file_results)}")
print()
print(f"{'Filename':<45} {'Status':<25} {'Archive key'}")
print("-" * 100)
for filename, fr in result.file_results.items():
    print(f"{filename:<45} {fr.upload_status:<25}  {fr.archive_key or ''}")

Dry run : False
Files   : 16

Filename                                      Status                    Archive key
----------------------------------------------------------------------------------------------------
pdb_chain_cath_uniprot.tsv.gz                 new                        
pdb_chain_ensembl.tsv.gz                      new                        
pdb_chain_enzyme.tsv.gz                       new                        
pdb_chain_go.tsv.gz                           new                        
pdb_chain_hmmer.tsv.gz                        new                        
pdb_chain_interpro.tsv.gz                     new                        
pdb_chain_pfam.tsv.gz                         new                        
pdb_chain_scop2_uniprot.tsv.gz                new                        
pdb_chain_scop2b_sf_uniprot.tsv.gz            new                        
pdb_chain_scop_uniprot.tsv.gz                 new                        
pdb_chain_taxonomy.tsv.gz                    